# 20 · Pruebas de carga del servicio EmoGol

Este notebook mide cómo se comporta EmoGol cuando aumenta la cantidad de mensajes, como pide la actividad 11 del anteproyecto: condiciones normales, alta demanda y niveles cercanos a la saturación.

**Cómo funciona**

1. Se levanta el servicio de inferencia (`api/main.py`), que carga `emogol.joblib` una sola vez y agrupa los mensajes en lotes de hasta 16.
2. Un generador de carga envía mensajes a una tasa fija (mensajes por segundo), sin esperar la respuesta del anterior, como llegan en un chat en vivo.
3. La tasa sube por niveles. En cada nivel se miden el rendimiento logrado, la latencia (p50, p95, p99), la cola máxima, el tamaño medio de lote y el uso de GPU.
4. Un nivel está **saturado** cuando el servicio clasifica menos del 90 % de lo que recibe, la latencia crece durante el nivel (la cola se acumula) o aparecen errores.

**Escenarios**

- *Chat:* los 269 mensajes reales del chat en vivo de la final Sub-20 (mensajes cortos).
- *Comentarios:* 2.000 comentarios de la prueba final (más largos, como los de un video).

La tasa real del chat fue de 1,12 mensajes/s, que se toma como condición normal.

**Entorno:** Google Colab con GPU T4, la misma de la evaluación. Duración aproximada: 50 minutos.
**Secretos:** `HF_TOKEN`.
**Resultado:** tablas y gráficas en `emotion-futbol-data/prototipo/pruebas_carga/`.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes "huggingface_hub<2.0" "pandas<3" joblib fastapi uvicorn httpx
!pip uninstall -y -q torchao
!rm -rf emotion_futbol && git clone -q https://github.com/santiagortegasegura/emotion_futbol.git
!pip install -q -e emotion_futbol/emogol_app

In [ ]:
import os, sys
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

USUARIO_HF = "santiagortegasegura"
REPO_MODELO = f"{USUARIO_HF}/emogol-2b"
REPO_DATOS = f"{USUARIO_HF}/emotion-futbol-data"
APP = "/content/emotion_futbol/emogol_app"
sys.path.insert(0, APP)

TASAS_CHAT = [0.5, 1.12, 2, 3, 4, 5, 6, 8, 10]     # mensajes por segundo
TASAS_COMENTARIOS = [0.5, 1, 2, 3, 4, 5, 6]
DURACION_NIVEL = 120                               # segundos de envío por nivel
PUERTO = 8000

## 1. Levantar el servicio con `emogol.joblib`

El archivo se descarga del repositorio `emogol-2b` (notebook 19). El servicio corre en segundo plano en la misma máquina, como lo haría en un contenedor en la nube.

In [ ]:
import subprocess, time, httpx
from huggingface_hub import hf_hub_download

ruta_joblib = hf_hub_download(REPO_MODELO, "emogol.joblib")
entorno = {**os.environ, "EMOGOL_JOBLIB": ruta_joblib, "PYTHONPATH": APP}
servidor = subprocess.Popen(["python", "-m", "uvicorn", "api.main:app", "--host", "127.0.0.1", "--port", str(PUERTO)],
                            cwd=APP, env=entorno, stdout=open("servidor.log", "w"), stderr=subprocess.STDOUT)
URL = f"http://127.0.0.1:{PUERTO}"
for _ in range(120):
    try:
        salud = httpx.get(f"{URL}/salud", timeout=2).json(); break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError(open("servidor.log").read()[-2000:])
print(salud)

## 2. Mensajes de prueba

In [ ]:
import pandas as pd

chat = pd.read_csv(hf_hub_download(REPO_DATOS, "prototipo/en_vivo_prueba/en_vivo_prueba_chat_clasificado.csv",
                                   repo_type="dataset"), keep_default_na=False)
prueba = pd.read_csv(hf_hub_download(REPO_DATOS, "final/prueba_final_10k.csv", repo_type="dataset"),
                     keep_default_na=False)
textos_chat = chat["text"].astype(str).tolist()
textos_comentarios = prueba.sample(2000, random_state=42)["text"].astype(str).tolist()
print(f"Chat: {len(textos_chat)} mensajes (mediana {chat['text'].str.len().median():.0f} caracteres)")
print(f"Comentarios: {len(textos_comentarios)} (mediana {pd.Series(textos_comentarios).str.len().median():.0f} caracteres)")

## 3. Latencia de un mensaje aislado

Referencia sin carga: se envía un mensaje, se espera la respuesta y se envía el siguiente.

In [ ]:
import statistics
lat = []
with httpx.Client(timeout=60) as cliente:
    for texto in textos_chat[:50]:
        t0 = time.perf_counter()
        cliente.post(f"{URL}/clasificar", json={"textos": [texto], "fuente": "chat"})
        lat.append(1000 * (time.perf_counter() - t0))
print(f"Latencia uno a uno: mediana {statistics.median(lat):.0f} ms | p95 {sorted(lat)[int(0.95 * len(lat))]:.0f} ms")

## 4. Barrido de carga: chat en vivo

In [ ]:
from pruebas_carga.generador import barrido, clasificar_niveles

resultados_chat = clasificar_niveles(await barrido(URL, textos_chat, TASAS_CHAT, DURACION_NIVEL, "chat"))
resultados_chat

## 5. Barrido de carga: comentarios de video

In [ ]:
resultados_com = clasificar_niveles(await barrido(URL, textos_comentarios, TASAS_COMENTARIOS, DURACION_NIVEL,
                                                 "comentarios"), tasa_real=1.12)
resultados_com

## 6. Resumen y gráficas

In [ ]:
import json, torch, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

ORO, AZUL, TINTA, REJILLA = "#A8841F", "#4A7EBF", "#3B3B3B", "#E4E3DF"
resultados = pd.concat([resultados_chat, resultados_com], ignore_index=True)

def capacidad(df):
    return float(df["rendimiento_logrado"].max())

def primer_saturado(df):
    s = df[df["saturado"]]
    return float(s["tasa_ofrecida"].min()) if len(s) else None

resumen = {
    "gpu": torch.cuda.get_device_name(0),
    "latencia_uno_a_uno_ms": {"mediana": round(statistics.median(lat), 1), "p95": round(sorted(lat)[int(0.95 * len(lat))], 1)},
    "chat": {"capacidad_msg_s": capacidad(resultados_chat), "tasa_saturacion_msg_s": primer_saturado(resultados_chat),
             "veces_tasa_real": round(capacidad(resultados_chat) / 1.12, 1)},
    "comentarios": {"capacidad_msg_s": capacidad(resultados_com), "tasa_saturacion_msg_s": primer_saturado(resultados_com)},
    "duracion_nivel_s": DURACION_NIVEL,
}
print(json.dumps(resumen, indent=2, ensure_ascii=False))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
for df, color, nombre in [(resultados_chat, ORO, "Chat en vivo"), (resultados_com, AZUL, "Comentarios de video")]:
    a1.plot(df["tasa_ofrecida"], df["rendimiento_logrado"], "o-", color=color, lw=2, label=nombre)
    a2.plot(df["tasa_ofrecida"], df["latencia_p95_ms"] / 1000, "o-", color=color, lw=2, label=nombre)
lim = max(TASAS_CHAT)
a1.plot([0, lim], [0, lim], "--", color="#9A9A9A", lw=1, label="Rendimiento ideal")
a1.axvline(1.12, color=TINTA, lw=0.8, ls=":"); a1.text(1.2, 0.3, "tasa real del chat", fontsize=8, color=TINTA)
a1.set_xlabel("Mensajes ofrecidos por segundo"); a1.set_ylabel("Mensajes clasificados por segundo")
a2.set_xlabel("Mensajes ofrecidos por segundo"); a2.set_ylabel("Latencia p95 (s)"); a2.set_yscale("log")
a2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
a2.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: ""))
for a in (a1, a2):
    a.grid(color=REJILLA); a.spines[["top", "right"]].set_visible(False); a.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig("pruebas_carga.png", dpi=200); plt.show()

## 7. Guardar y publicar

In [ ]:
from huggingface_hub import upload_file

resultados.to_csv("pruebas_carga_resultados.csv", index=False)
json.dump(resumen, open("pruebas_carga_resumen.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
for archivo in ["pruebas_carga_resultados.csv", "pruebas_carga_resumen.json", "pruebas_carga.png", "servidor.log"]:
    upload_file(path_or_fileobj=archivo, path_in_repo=f"prototipo/pruebas_carga/{archivo}",
                repo_id=REPO_DATOS, repo_type="dataset")
    print(f"Subido: prototipo/pruebas_carga/{archivo}")
servidor.terminate()